# Bollywood Nepotism Analysis, 2000–2025

**Goal:** measure how much of the Bollywood release slate (and its box-office revenue) is driven by star kids.

## Pipeline
1. **Scrape** Wikipedia's *List of Bollywood films of YYYY* pages for every release 2000–2025 → `bollywood_all_2000_2025.csv` (run via `scrape_bollywood_all_2000_2025.py`)
2. **Fill missing gross** from per-film Wikipedia infoboxes → `bollywood_filled_gross.csv` (run via `fill_gross_from_wiki.py`)
3. **Tag** each film with `nepo_kid = yes/no` using a curated list of first-generation children of prominent film-industry figures
4. **Aggregate** by year, by actor, by director
5. **Visualise**

## Caveats
- *Gross* is missing for most non-hit films even after the per-film fill — Wikipedia only records box-office for notable releases.
- *Nepo kid* is a curated definition (first-generation children/close relatives of prominent film figures). Edit `NEPO_KIDS` below to refine.
- All amounts in ₹ **crore** (1 crore = 10 million).

## 1. Setup & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", 80)
plt.rcParams["figure.figsize"] = (11, 5)

# Prefer the filled-gross version if it exists
IN_PATH = Path("bollywood_filled_gross.csv")
if not IN_PATH.exists():
    IN_PATH = Path("bollywood_all_with_nepo.csv")
print(f"Loading {IN_PATH}")

df = pd.read_csv(IN_PATH)
df["year"] = df["year"].astype(int)
df["gross_cr"] = pd.to_numeric(df["gross_cr"], errors="coerce")
print(f"{len(df)} films, {df['year'].min()}-{df['year'].max()}")
print(f"With gross: {df['gross_cr'].notna().sum()}")
df.head()

## 2. Nepo-kid list (curated)

In [ ]:
NEPO_KIDS = {
    # Bachchan family
    "Abhishek Bachchan", "Agastya Nanda", "Navya Naveli Nanda",
    # Kapoor family (Raj Kapoor lineage + Surinder + Jeetendra + Pankaj Kapur)
    "Karisma Kapoor", "Karishma Kapoor",
    "Kareena Kapoor", "Kareena Kapoor Khan",
    "Ranbir Kapoor", "Rishi Kapoor",
    "Tusshar Kapoor", "Shahid Kapoor",
    "Sonam Kapoor", "Arjun Kapoor", "Anshula Kapoor",
    "Janhvi Kapoor", "Khushi Kapoor", "Shanaya Kapoor",
    "Aadar Jain", "Armaan Jain", "Sanjay Kapoor",
    "Ishaan Khatter",
    # Roshan family
    "Hrithik Roshan", "Pashmina Roshan",
    # Salim Khan / Salman family
    "Salman Khan", "Sohail Khan", "Arbaaz Khan", "Arhaan Khan",
    # Tahir Hussain / Aamir Khan family
    "Aamir Khan", "Faisal Khan", "Imran Khan", "Junaid Khan", "Ira Khan",
    # Pataudi-Tagore family (Saif)
    "Saif Ali Khan", "Soha Ali Khan", "Sara Ali Khan", "Ibrahim Ali Khan",
    # Shah Rukh Khan family
    "Aryan Khan", "Suhana Khan",
    # Deol / Dharmendra family
    "Sunny Deol", "Bobby Deol", "Esha Deol",
    "Karan Deol", "Rajveer Deol", "Abhay Deol",
    # Dutt family
    "Sanjay Dutt", "Trishala Dutt",
    # Khanna families
    "Twinkle Khanna", "Akshaye Khanna", "Rinke Khanna",
    # Yash Chopra family
    "Uday Chopra",
    # Mukherjee-Samarth family
    "Rani Mukerji", "Kajol", "Tanishaa Mukerji", "Mohnish Bahl",
    # Sinha family
    "Sonakshi Sinha", "Luv Sinha", "Kussh Sinha",
    # Oberoi family
    "Vivek Oberoi",
    # Ganesan family
    "Rekha",
    # Bhatt family
    "Alia Bhatt", "Pooja Bhatt", "Emraan Hashmi", "Rahul Bhatt",
    # Dhawan family
    "Varun Dhawan", "Rohit Dhawan",
    # Shetty (Suniel) family
    "Athiya Shetty", "Ahan Shetty",
    # Shroff (Jackie) family
    "Tiger Shroff", "Krishna Shroff",
    # Panday (Chunky) family
    "Ananya Panday", "Rysa Panday",
    # Roy Kapur family
    "Aditya Roy Kapur", "Kunaal Roy Kapur",
    # Kaushal (Sham) family
    "Vicky Kaushal", "Sunny Kaushal",
    # Irrfan Khan family
    "Babil Khan", "Ayaan Khan",
    # Devgan family
    "Aaman Devgan", "Nysa Devgan",
    # Tandon family
    "Rasha Thadani",
    # Babbar family
    "Pratik Babbar", "Juhi Babbar", "Arya Babbar",
}
print(f"Curated nepo-kid list: {len(NEPO_KIDS)} actors")

## 3. Tag films and explode cast

Two derived tables:
- `films`  — one row per film with `nepo_kid` flag and matched names
- `long`   — one row per (actor, film), ready for time-series analysis

In [ ]:
def split_cast(s):
    if pd.isna(s) or not s:
        return []
    return [c.strip() for c in str(s).split(";") if c.strip()]

films = df.copy()
films["cast_list"] = films["cast"].apply(split_cast)
films["nepo_in_cast"] = films["cast_list"].apply(
    lambda lst: [a for a in lst if a in NEPO_KIDS]
)
films["nepo_kid"] = films["nepo_in_cast"].apply(lambda hits: "yes" if hits else "no")
films["n_nepo_in_cast"] = films["nepo_in_cast"].apply(len)

# Long format: one row per actor-film
long = films.explode("cast_list").rename(columns={"cast_list": "actor"})
long = long[long["actor"].notna() & (long["actor"] != "")].copy()
long["actor_nepo"] = long["actor"].isin(NEPO_KIDS).map({True: "yes", False: "no"})
long = long[["actor", "actor_nepo", "year", "title", "gross_cr",
             "nepo_kid", "director"]].rename(columns={"nepo_kid": "film_has_nepo_cast",
                                                       "title": "movie"})

print(f"films:  {len(films)} rows")
print(f"long:   {len(long)} actor-film rows ({long['actor'].nunique()} unique actors)")
print(f"\nFilms with any nepo kid: {(films['nepo_kid']=='yes').sum()} "
      f"({(films['nepo_kid']=='yes').mean()*100:.1f}%)")

## 4. Yearly aggregates

In [ ]:
yearly = (
    films.groupby(["year", "nepo_kid"])
         .agg(films=("title", "count"),
              total_gross_cr=("gross_cr", "sum"),
              avg_gross_cr=("gross_cr", "mean"),
              films_with_gross=("gross_cr", "count"))
         .unstack("nepo_kid")
         .fillna(0)
)
yearly.columns = [f"{stat}_{flag}" for stat, flag in yearly.columns]
yearly["total_films"] = yearly["films_yes"] + yearly["films_no"]
yearly["nepo_share_films_pct"] = (yearly["films_yes"] / yearly["total_films"] * 100).round(1)
yearly["nepo_share_gross_pct"] = (
    yearly["total_gross_cr_yes"] / (yearly["total_gross_cr_yes"] + yearly["total_gross_cr_no"]) * 100
).round(1)
yearly

## 5. Charts

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

# (a) Number of films per year, stacked
ax[0].bar(yearly.index, yearly["films_no"], label="Non-nepo", color="#888")
ax[0].bar(yearly.index, yearly["films_yes"], bottom=yearly["films_no"],
          label="Has nepo kid", color="#c0392b")
ax[0].set_ylabel("Films released")
ax[0].set_title("Bollywood releases per year: nepo-cast vs non-nepo")
ax[0].legend(loc="upper left")

# (b) Share lines
ax[1].plot(yearly.index, yearly["nepo_share_films_pct"], "o-", label="% of films", color="#c0392b")
ax[1].plot(yearly.index, yearly["nepo_share_gross_pct"], "s--",
           label="% of gross (only films with gross data)", color="#2980b9")
ax[1].set_ylabel("Nepo share (%)")
ax[1].set_xlabel("Year")
ax[1].set_title("Nepo share of films vs of box-office gross")
ax[1].axhline(50, color="#aaa", linestyle=":", linewidth=0.8)
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Average gross per film, nepo vs non-nepo (only films with gross data)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(yearly.index, yearly["avg_gross_cr_yes"], "o-", label="Nepo-cast films", color="#c0392b")
ax.plot(yearly.index, yearly["avg_gross_cr_no"], "s-", label="Non-nepo films", color="#2980b9")
ax.set_ylabel("Avg gross per film (₹ crore)")
ax.set_xlabel("Year")
ax.set_title("Average gross per film, nepo vs non-nepo (among films with gross data)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Per-actor analysis

In [ ]:
actor_stats = (
    long.groupby("actor")
        .agg(films=("movie", "count"),
             total_gross_cr=("gross_cr", "sum"),
             films_with_gross=("gross_cr", "count"),
             first_year=("year", "min"),
             last_year=("year", "max"))
        .reset_index()
)
actor_stats["nepo"] = actor_stats["actor"].isin(NEPO_KIDS).map({True: "yes", False: "no"})
actor_stats["avg_gross_cr"] = (
    actor_stats["total_gross_cr"] / actor_stats["films_with_gross"].replace(0, np.nan)
).round(2)
actor_stats = actor_stats.sort_values("films", ascending=False)
actor_stats.head(20)

In [ ]:
# Top 20 by total gross (nepo highlighted)
top_gross = actor_stats.sort_values("total_gross_cr", ascending=False).head(20)
colors = ["#c0392b" if n == "yes" else "#2980b9" for n in top_gross["nepo"]]
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(top_gross["actor"][::-1], top_gross["total_gross_cr"][::-1], color=colors[::-1])
ax.set_xlabel("Total gross across all films (₹ crore)")
ax.set_title("Top 20 actors by total box-office (red = nepo kid)")
plt.tight_layout()
plt.show()

In [ ]:
# Time series: top 6 actors by film count, films per year
top_actors = actor_stats.head(6)["actor"].tolist()
ts = (long[long["actor"].isin(top_actors)]
      .groupby(["actor", "year"]).size().unstack(fill_value=0)
      .reindex(columns=range(films["year"].min(), films["year"].max() + 1), fill_value=0))

fig, ax = plt.subplots(figsize=(11, 5))
for actor in top_actors:
    nepo_marker = "o-" if actor in NEPO_KIDS else "s--"
    ax.plot(ts.columns, ts.loc[actor], nepo_marker, label=actor, alpha=0.85)
ax.set_xlabel("Year")
ax.set_ylabel("Films released")
ax.set_title("Annual film count: top 6 most prolific actors")
ax.legend(loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Wide per-actor time series (export)

In [ ]:
# Wide: actor | nepo | num_films | total_gross_cr | year_1 | movie_1 | gross_cr_1 | ...
rows = []
for actor, grp in long.sort_values(["actor", "year", "movie"]).groupby("actor"):
    row = {
        "actor": actor,
        "nepo": "yes" if actor in NEPO_KIDS else "no",
        "num_films": len(grp),
        "total_gross_cr": round(grp["gross_cr"].sum(skipna=True), 2),
        "films_with_gross": int(grp["gross_cr"].notna().sum()),
    }
    for i, (_, r) in enumerate(grp.iterrows(), 1):
        row[f"year_{i}"] = r["year"]
        row[f"movie_{i}"] = r["movie"]
        row[f"gross_cr_{i}"] = round(r["gross_cr"], 2) if pd.notna(r["gross_cr"]) else ""
    rows.append(row)

wide = pd.DataFrame(rows).sort_values(["num_films", "actor"], ascending=[False, True])
OUT = "actor_timeseries_wide_notebook.csv"
wide.to_csv(OUT, index=False)
print(f"Wrote {len(wide)} actors to {OUT}")
wide.head(10)

## 8. Director analysis (bonus)

In [ ]:
dir_stats = (
    films.assign(director=films["director"].fillna("").str.strip())
         .query("director != ''")
         .groupby("director")
         .agg(films=("title", "count"),
              nepo_films=("nepo_kid", lambda s: (s == "yes").sum()),
              total_gross_cr=("gross_cr", "sum"))
         .reset_index()
)
dir_stats["nepo_share_pct"] = (dir_stats["nepo_films"] / dir_stats["films"] * 100).round(1)
dir_stats = dir_stats[dir_stats["films"] >= 5].sort_values("nepo_share_pct", ascending=False)
dir_stats.head(20)

## Notes / next steps

- Editing the curated `NEPO_KIDS` set above and re-running cells 3–8 will refresh everything end-to-end.
- If the background `fill_gross_from_wiki.py` job finishes after you ran this notebook, restart the kernel — it'll auto-pick up `bollywood_filled_gross.csv` in cell 1.
- For causal claims about nepotism (talent vs access), this dataset is observational only. A proper analysis would need: comparable opening-weekend slates, debut-budget-controlled comparisons, and inclusion of unreleased / direct-to-OTT films.